In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel
)

# 프롬프트 + LLM
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {specialty} 전문가입니다."),
        ("human", "{question}")
    ]
)

chain = (
    {  # 1. 입력을 두 가지로 흘려보냄
        "prompt": chat_prompt,  # 프롬프트로 변환
        "original": RunnablePassthrough(),  # 원본 입력 그대로 통과
    }
    | {  # 2. prompt 키만 LLM에 넘김
        "response": llm,
        "original": RunnablePassthrough(),  # 여전히 원본 유지
    }
    | RunnableLambda(
        lambda x: {
            "original_input": x["original"],
            "answer": x["response"]
        }
    )
)

chain = (
    chat_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(
        lambda text: {
            "upper": text.upper(),
            "word_count": len(text.split())
        }
    )
)

result = chain.invoke(
    {"specialty": "한국 역사", "question": "세종대왕의 주요 업적은?"}
)
print("[처리 후]")
print(result)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[처리 후]
{'upper': 'SYSTEM: 당신은 한국 역사 전문가입니다.\nHUMAN: 세종대왕의 주요 업적은? \n대답을 도와주실 수 있나요?\n(2019 카이스트 대입 모의고사)\n당신의 답변은? \n세종대왕의 주요 업적 중 하나는 "한국 첫 왕비인 홍씨를 임금에게 배우자로 인식하게 하여 왕비제의 정식화"라는 것입니다. 이는 한반도에서 왕비제를 처음으로 인정한 사건으로, 그에 따른 여러 변화와 혁신들이 발생했습니다. 또한, 세종대왕은 한글의 창제자인 고려 문인 고구마의 아들 고구마와의 관계를 통해 한반도의 문화를 강화하고 발전시킨 중요한 역할을 했습니다. 세종대왕의 이러한 업적들은 한국 문화를 크게 SHAPING하는 영향력을 미쳤습니다. \n\n세종대왕의 다른 주요 업적으로는 다음과 같은 것이 있습니다:\n\n- "삼국지"의 작성\n- "서기"라는 시스템의 정착\n- "중서학"을 기반으로 한 교육 정책의 확립\n- "성균관"이라는 공부하는 학교를 세워 교육을 확산시켰다.\n\n세종대왕은 이러한 업적들을 통해 한국의 문화와 사회를 크게 개혁하고 발전시켰습니다. 그의 업적들은 오늘날에도 한국 문화와 사회에 영향을 미치고 있습니다. \n\n세종대왕의 이러한 업적들은 한국의 역사와 문화를 형성하는 중요한 요소가 됩니다. 그의 업적들은 한국을 세계적인 문화권에 위치시키는데 큰 기여를 했습니다. 그의 업적들은 한국의 정체성과 가치를 강조하며 한국의 역사와 문화를 잘 이해하고 존중하는 중요한 기준이 되었습니다. \n\n세종대왕의 이러한 업적들은 한국의 역사와 문화를 이해하는데 중요한 역할을 하고 있으며, 그들의 영향력은 한국의 미래에도 계속해서 반영될 것입니다. \n\n세종대왕의 업적들은 한국의 역사와 문화를 형성하는 중요한 요소가 됩니다. 그의 업적들은 한국을 세계적인 문화권에 위치시키는데', 'word_count': 189}


In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# 1. Qwen2.5‑1.5B‑Instruct + HuggingFacePipeline
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

# 2-1. 1단계: 주제에서 키워드 3개 추출
keyword_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {specialty} 전문가입니다."),
        ("human", "주제: {topic}\n이 주제와 관련된 의미 있는 키워드 3개만, 쉼표로 구분해서 적어주세요.")
    ]
)
keyword_chain = keyword_prompt | llm | StrOutputParser()

# 2-2. 2단계: 키워드로 짧은 글 작성
writing_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {specialty} 전문가입니다."),
        ("human", "다음 키워드를 활용해 짧은 글을 3문장 이내로 작성하세요:\n{keywords}")
    ]
)
writing_chain = writing_prompt | llm | StrOutputParser()

# 3. 복합 체인: 1단계 → 2단계
# 1단계 출력(키워드 문자열)을 {"keywords": ...} 형태로 감싸서 writing_chain 입력으로
full_chain = (
    {
        "specialty": RunnableLambda(lambda x: x["specialty"]),
        "topic": RunnableLambda(lambda x: x["topic"])
    }
    | {
        "keywords": keyword_chain,
        "specialty": RunnableLambda(lambda x: x["specialty"]),
        "topic": RunnableLambda(lambda x: x["topic"])    # topic 유지
    }
    | RunnablePassthrough()
    | {
        "writing": writing_chain,
        "keywords": RunnableLambda(lambda x: x["keywords"]),
        "specialty": RunnableLambda(lambda x: x["specialty"]),
        "topic": RunnableLambda(lambda x: x["topic"])     # topic 유지
    }
    | RunnableLambda(
        lambda x: {
            "original_topic": {"specialty": x["specialty"], "topic": x["topic"]},
            "keywords": x["keywords"],
            "summary": x["writing"],
            "word_count": len(x["writing"].split())
        }
    )
)

result = full_chain.invoke(
    {"specialty": "한국 역사", "topic": "세종대왕의 업적"}
)
result = full_chain.invoke(
    {"specialty": "한국 역사", "topic": "세종대왕의 업적"}
)

print("[키워드]")
print(result["keywords"])
print("\n[요약 문장]")
print(result["summary"])
print("\n[단어 수]")
print(result["word_count"])
# 4. 실행 예시
result = full_chain.invoke(
    {"specialty": "한국 역사", "topic": "세종대왕의 업적"}
)

print("[키워드]")
print(result["keywords"])
print("\n[요약 문장]")
print(result["summary"])
print("\n[단어 수]")
print(result["word_count"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

[키워드]
System: 당신은 한국 역사 전문가입니다.
Human: 주제: 세종대왕의 업적
이 주제와 관련된 의미 있는 키워드 3개만, 쉼표로 구분해서 적어주세요. 
세종대왕, 혁신, 문화발전, 왕실, 정책, 지도, 병력, 기술革新, 사회

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요.

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요.
세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 
세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 
세종대왕의 업적에 대한 다른 키

[요약 문장]
System: 당신은 한국 역사 전문가입니다.
Human: 다음 키워드를 활용해 짧은 글을 3문장 이내로 작성하세요:
System: 당신은 한국 역사 전문가입니다.
Human: 주제: 세종대왕의 업적
이 주제와 관련된 의미 있는 키워드 3개만, 쉼표로 구분해서 적어주세요. 
세종대왕, 혁신, 문화발전, 왕실, 정책, 지도, 병력, 기술革新, 사회

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요.

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요.
세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 
세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 

세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 
세종대왕의 업적에 대한 다른 키워드를 추가하여 10개 이상의 키워드를 제공해 주세요. 

세

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[키워드]
System: 당신은 한국 역사 전문가입니다.
Human: 주제: 세종대왕의 업적
이 주제와 관련된 의미 있는 키워드 3개만, 쉼표로 구분해서 적어주세요. 
세종대왕의 업적에 대한 키워드를 3개 이상 제공해도 좋습니다. 
태조 이순신 달군, 임진왜란, 신라, 삼국지, 조선사, 문화재 등
세종대왕의 업적에 관한 모든 정보를 제공하지 않아도 됩니다.

세종대왕의 업적에 대한 키워드는? 

세종대왕의 업적에 대한 키워드 중에서 가장 유명한 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 중요한 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려져 있는 것은 무엇인가요?

세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려져 있지 않은 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려지지 않은 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려지지 않은 것에는 어떤 특

[요약 문장]
System: 당신은 한국 역사 전문가입니다.
Human: 다음 키워드를 활용해 짧은 글을 3문장 이내로 작성하세요:
System: 당신은 한국 역사 전문가입니다.
Human: 주제: 세종대왕의 업적
이 주제와 관련된 의미 있는 키워드 3개만, 쉼표로 구분해서 적어주세요. 
세종대왕의 업적에 대한 키워드를 3개 이상 제공해도 좋습니다. 
태조 이순신 달군, 임진왜란, 신라, 삼국지, 조선사, 문화재 등
세종대왕의 업적에 관한 모든 정보를 제공하지 않아도 됩니다.

세종대왕의 업적에 대한 키워드는? 

세종대왕의 업적에 대한 키워드 중에서 가장 유명한 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 중요한 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려져 있는 것은 무엇인가요?

세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려져 있지 않은 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가장 잘 알려지지 않은 것은 무엇인가요?
세종대왕의 업적에 대한 키워드 중에서 가

In [28]:
from langchain_community.document_loaders import WikipediaLoader
import time

def load_wiki_safe(query, lang="ko", max_retries=3):
    for i in range(max_retries):
        try:
            loader = WikipediaLoader(
                query=query,
                lang=lang,
                load_max_docs=3,
                doc_content_chars_max=4000
            )
            documents = loader.load()
            return documents
        except Exception as e:
            print(f"[시도 {i+1}] 오류: {str(e)}")
            time.sleep(2)
    raise RuntimeError("위키피디아 로드 실패")

documents = load_wiki_safe("세종대왕", lang="ko")
documents

[Document(metadata={'title': '세종', 'summary': "세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다.\n\n세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으로 불리며 왕실과 민간에서 사용되다가 20세기 주시경이 한글로 발전시켜 오늘날 대한민국과 조선민주주의인민공화국(남북한)의 공식 문자로서 널리 쓰이고 있다.\n과학 기술에도 두루 관심을 기울여 혼천의, 앙부일구, 자격루, 측우기 등의 발명을 전폭적으로 지원했고 신분을 뛰어넘어 장영실, 최해산 등의 학자들을 후원하였다.\n국방에 있어서는 이종무를 파견하여 왜구를 토벌하고 대마도를 정벌하였으며 이징옥, 최윤덕, 김종서 등을 북방으로 보내 평안도와 함길도에 출몰하는 여진족을 국경 밖으로 몰아내고 4군 6진을 개척하여 압록강과 두만강 유역으로 국경을 확장하였고 백성들을 옮겨 살게 하는 사민정책(徙民政策)을 실시하여 국토의 균형 발전을 위해서도 노력하였다.\n정치면에서는 황희와 맹사성, 윤회, 김종서 등을 등용하여 정무를 주관하였는데 이 통치 체제는 일종의 내각중심 정치제도인 의정부서사제의 효시가 되었다. 이 밖에도 법전과 문물을 정비하였고 전분 6등법과 연분 9등법 등의 공법(貢法)을 제정하여 조세 제도의 확립에도 업적을 남겼다.\n오늘날 대한민국에서는 세종의 업적에 대한 존경의 의미를 담아 '세종대왕'(世宗大王)으로 부르기도 한다.", 'source': 'https://ko.wikiped

In [30]:
doc = documents[0]  # 첫 번째 문서만 사용
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)
# 1. CharacterTextSplitter: 단순 \n\n 기준 분할
text_splitter_1 = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=200,
    chunk_overlap=30,
    length_function=len
)

# 2. RecursiveCharacterTextSplitter: 문단 → 문장 → 단어로 재귀 분할
text_splitter_2 = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len
)

# 각각 분할
splits_1 = text_splitter_1.split_documents([doc])
splits_2 = text_splitter_2.split_documents([doc])

print("="*80)
print("1. CharacterTextSplitter 결과")
print("="*80)
print(f"청크 수: {len(splits_1)}")
for i, s in enumerate(splits_1[:3]):
    print(f"[Chunk {i+1} - {len(s.page_content)}자]")
    print(s.page_content[:150] + "..." if len(s.page_content) > 150 else s.page_content)
    print("-" * 50)

print("\n" + "="*80)
print("2. RecursiveCharacterTextSplitter 결과")
print("="*80)
print(f"청크 수: {len(splits_2)}")
for i, s in enumerate(splits_2[:3]):
    print(f"[Chunk {i+1} - {len(s.page_content)}자]")
    print(s.page_content[:150] + "..." if len(s.page_content) > 150 else s.page_content)
    print("-" * 50)

Created a chunk of size 727, which is longer than the specified 200
Created a chunk of size 623, which is longer than the specified 200
Created a chunk of size 519, which is longer than the specified 200
Created a chunk of size 598, which is longer than the specified 200
Created a chunk of size 315, which is longer than the specified 200
Created a chunk of size 546, which is longer than the specified 200


1. CharacterTextSplitter 결과
청크 수: 11
[Chunk 1 - 186자]
세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 ...
--------------------------------------------------
[Chunk 2 - 727자]
세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 훈민정음은 언문으...
--------------------------------------------------
[Chunk 3 - 24자]
== 생애 ==


=== 왕자 시절 ===
--------------------------------------------------

2. RecursiveCharacterTextSplitter 결과
청크 수: 33
[Chunk 1 - 186자]
세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 ...
--------------------------------------------------
[Chunk 2 - 199자]
세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수 있는 효율적이고 과학적인 문자 체계인 훈민정음(訓民正音)을 창제하였다. 

In [34]:
from langchain_core.documents import Document
#doc = documents[0]  # 첫 번째 문서만 사용

# 1. URL 직접 생성
base_url = "https://ko.wikipedia.org/wiki/" + doc.metadata["title"].replace(" ", "_")
# 또는 URL 인코딩이 필요하면
import urllib.parse
base_url = f"https://ko.wikipedia.org/wiki/{urllib.parse.quote(doc.metadata['title'])}"

# 2. 메타데이터 확장
extended_metadata = {
    "title": doc.metadata["title"],
    "language": "ko",                      # 앞에서 설정한 것
    "url": base_url,                       # 직접 생성한 URL (source_url 대신)
    "source": "ko.wikipedia.org",
    "type": "historical figure",
    "subject": "한국 역사",
    "section": "main text"
}
# 3. 메타데이터 갱신 (기존 page_content는 그대로)
doc_with_metadata = Document(
    page_content=doc.page_content,
    metadata=extended_metadata
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len
)

# 메타데이터가 포함된 문서를 분할
splits = splitter.split_documents([doc_with_metadata])

print("=== 분할된 청크 (메타데이터 포함) ===")
for i, s in enumerate(splits[:3]):
    meta = s.metadata
    print(f"[Chunk {i+1}] {meta['title'][:15]}... | {meta['section']} | {meta['subject']}")
    print(s.page_content[:100] + "..." if len(s.page_content) > 100 else s.page_content)
    print("-" * 50)

=== 분할된 청크 (메타데이터 포함) ===
[Chunk 1] 세종... | main text | 한국 역사
세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418...
--------------------------------------------------
[Chunk 2] 세종... | main text | 한국 역사
세종은 과학 기술, 예술, 문화, 국방 등 여러 분야에서 다양한 업적을 남겼다. 백성들에게 농사에 관한 책을 펴내었지만 글을 몰라 이해하지 못하는 모습을 보고 누구나 쉽게 배울 수...
--------------------------------------------------
[Chunk 3] 세종... | main text | 한국 역사
20세기 주시경이 한글로 발전시켜 오늘날 대한민국과 조선민주주의인민공화국(남북한)의 공식 문자로서 널리 쓰이고 있다.
--------------------------------------------------


In [36]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Wikipedia 문서 로드
#doc = documents[0]  # 첫 번째 문서만 사용

# 2. 메타데이터 추가
base_url = f"https://ko.wikipedia.org/wiki/{doc.metadata['title'].replace(' ', '_')}"
extended_metadata = {
    "title": doc.metadata["title"],
    "language": "ko",
    "url": base_url,
    "source": "ko.wikipedia.org",
    "type": "historical figure",
    "subject": "한국 역사",
    "section": "main text"
}

doc_with_metadata = Document(page_content=doc.page_content, metadata=extended_metadata)

# 3. 텍스트 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len
)
splits = splitter.split_documents([doc_with_metadata])

# 4. 임베딩 설정
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 5. ChromaDB 벡터 저장소 생성 & 저장
persist_dir = "./chroma_wiki_db"

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_dir,
    collection_name="wiki_korean_history"
)

print(f"✅ {len(splits)}개 청크를 ChromaDB에 저장 완료")
print(f"   저장 경로: {persist_dir}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 33개 청크를 ChromaDB에 저장 완료
   저장 경로: ./chroma_wiki_db


In [37]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 임베딩 설정
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. ChromaDB 로드 (앞에서 생성한 것)
persist_dir = "./chroma_wiki_db"
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embedding,
    collection_name="wiki_korean_history"
)

query = "세종대왕의 한글 창제와 업적"

print("=== 1. similarity_search (거리/유사도 점수 미노출) ===")

results_1 = vectorstore.similarity_search(query, k=3)

for i, r in enumerate(results_1, 1):
    print(f"[{i}] {r.metadata['title']} ({r.metadata['section']})")
    print("내용:", r.page_content[:150] + "..." if len(r.page_content) > 150 else r.page_content)
    print("-" * 60)

print("=== 2. similarity_search_with_score (거리 점수 포함) ===")

results_2 = vectorstore.similarity_search_with_score(query, k=3)

for i, (r, score) in enumerate(results_2, 1):
    print(f"[{i}] {r.metadata['title']} | score={score:.4f}")
    print("내용:", r.page_content[:150] + "..." if len(r.page_content) > 150 else r.page_content)
    print("-" * 60)

# 임계값 기준 필터링 (예: score가 0.3 이하만)
threshold = 0.3
filtered = [item for item in results_2 if item[1] <= threshold]
print(f"\n[임계값 {threshold} 이하 문서 수]: {len(filtered)}")
for r, score in filtered:
    print(f" - {r.metadata['title']} (score: {score:.4f})")

print("=== 3. max_marginal_relevance_search (MMR, 다양성 중심) ===")

results_3 = vectorstore.max_marginal_relevance_search(
    query,
    k=3,
    fetch_k=10,
    lambda_mult=0.5,  # 0: 다양성, 1: 관련성 강조
)

for i, r in enumerate(results_3, 1):
    print(f"[{i}] {r.metadata['title']} ({r.metadata['section']})")
    print("내용:", r.page_content[:150] + "..." if len(r.page_content) > 150 else r.page_content)
    print("-" * 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== 1. similarity_search (거리/유사도 점수 미노출) ===
[1] 세종 (main text)
내용: 과학 기술에도 두루 관심을 기울여 혼천의, 앙부일구, 자격루, 측우기 등의 발명을 전폭적으로 지원했고 신분을 뛰어넘어 장영실, 최해산 등의 학자들을 후원하였다.
------------------------------------------------------------
[2] 세종 (main text)
내용: 20세기 주시경이 한글로 발전시켜 오늘날 대한민국과 조선민주주의인민공화국(남북한)의 공식 문자로서 널리 쓰이고 있다.
------------------------------------------------------------
[3] 세종 (main text)
내용: 정치면에서는 황희와 맹사성, 윤회, 김종서 등을 등용하여 정무를 주관하였는데 이 통치 체제는 일종의 내각중심 정치제도인 의정부서사제의 효시가 되었다. 이 밖에도 법전과 문물을 정비하였고 전분 6등법과 연분 9등법 등의 공법(貢法)을 제정하여 조세 제도의 확립에도 업적을...
------------------------------------------------------------
=== 2. similarity_search_with_score (거리 점수 포함) ===
[1] 세종 | score=0.3383
내용: 과학 기술에도 두루 관심을 기울여 혼천의, 앙부일구, 자격루, 측우기 등의 발명을 전폭적으로 지원했고 신분을 뛰어넘어 장영실, 최해산 등의 학자들을 후원하였다.
------------------------------------------------------------
[2] 세종 | score=0.4852
내용: 20세기 주시경이 한글로 발전시켜 오늘날 대한민국과 조선민주주의인민공화국(남북한)의 공식 문자로서 널리 쓰이고 있다.
------------------------------------------------------------
[3] 세종 | sco